In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import numpy as np
import arc
from arc import PairStateInteractions
import pairinteraction as pi
if pi.Database.get_global_database() is None:
    pi.Database.initialize_global_database(download_missing=False)
import os
from IPython.display import Latex
import scipy
import pandas as pd
from pandas import DataFrame
import sys
from time import time
sys.path.append(os.path.abspath("C:\\Users\\Tommy\\OneDrive - Durham University\\level 4 Project\\Code Directory\\L4-code-directory-2"))
from my_functions import *
h = scipy.constants.h
epsilon_0 = scipy.constants.epsilon_0
a_0 = scipy.constants.physical_constants['Bohr radius']
e = scipy.constants.e


In [2]:
def get_system_pair_list_erange(
    ket1: pi.SystemAtom, ket2: pi.SystemAtom, theta, eranges, r, b, e, ndif, ldif, jdif
) -> list[pi.SystemPair]:


    basis1 = pi.BasisAtom(ket1.species, n=(ket1.n-ndif, ket1.n+ndif), l=(0,ket1.l+ldif), j=(1/2,ket1.j+jdif), m=(-ket1.j-jdif, ket1.j+jdif))
    basis2 = pi.BasisAtom(ket2.species, n=(ket2.n-ndif, ket2.n+ndif), l=(0,ket2.l+ldif), j=(1/2,ket2.j+jdif), m=(-ket2.j-jdif, ket2.j+jdif))



    system1, system2 = pi.SystemAtom(basis1).set_magnetic_field([0,0,b], unit='G').set_electric_field([0,0,e], unit='V/cm').set_diamagnetism_enabled(True), pi.SystemAtom(basis2).set_magnetic_field([0,0,b], unit='G').set_electric_field([0,0,e], unit='V/cm').set_diamagnetism_enabled(True)


    pi.diagonalize([system1, system2])

    pair_energy = system1.get_corresponding_energy(ket1, unit=energy_unit) + system2.get_corresponding_energy(ket2, unit=energy_unit)
    

    pair_systems = []
    state_nums = []

    # timing setup
    times = []
    
    # Loop over energies
    for i, erange in enumerate(eranges):
        t0 = time.time()

        min_energy, max_energy = pair_energy - erange, pair_energy + erange
        
        pair_basis = pi.BasisPair(
                    [system1, system2], energy=(min_energy, max_energy), energy_unit=energy_unit
                )

        pair_system = pi.SystemPair(pair_basis).set_distance(r, unit='um', angle_degree=theta).set_interaction_order(3)
        
        pair_systems.append(pair_system)

        state_nums.append(pair_basis.number_of_states)
        
        # Diagonalize the systems in individually - just for timing purposes
        pi.diagonalize(
            [pair_system],
            diagonalizer="eigen",
            energy_range=(min_energy, max_energy),
            energy_range_unit=energy_unit,
        )

        times.append(time.time() - t0)
    
    # Diagonalise in parallel for use - uses largest energy range here
    pi.diagonalize(
                pair_systems,
                diagonalizer="eigen",
                energy_range=(min_energy, max_energy),
                energy_range_unit=energy_unit,
            )
    
    energies = [system.get_eigenenergies(unit = energy_unit) - pair_energy for system in pair_systems]

    return pair_systems, state_nums, energies, times


In [3]:
energy_unit = 'MHz'
atom1, atom2 = 'Rb', 'Cs'
l1_0, l2_0 = 0, 0
j1_0, j2_0 = 1/2, 1/2
l1_1, l2_1 = 1, 1

channels = channels_import(atom1, l1_0, j1_0, atom2, l2_0, j2_0, l1_1, l2_1)
channel=5
c=channels.iloc[channel]

r = 5
theta=0
b=0
e=0

ndif, ldif, jdif = 4, 2, 2

n1_0, l1_0, j1_0 = int(c['n1_0']), int(c['l1_0']), c['j1_0']
n2_0, l2_0, j2_0 = int(c['n2_0']), int(c['l2_0']), c['j2_0']
mj1_0s, mj2_0s = np.arange(-j1_0, j1_0+1), np.arange(-j2_0, j2_0+1)
mj1, mj2 = mj1_0s[-1], mj2_0s[-1]
defect = c['defect']

eranges = np.arange(10, 1e5, 10000)

# Run file tag
run_tag = f'\\r {r}um, theta {theta}deg, b {b}G, nljdiff{ndif}{ldif}{jdif}, erange {eranges[-1]-eranges[0]}MHz'
print(run_tag)

# run log data paths
runlog_datapath = fr'E:\\Tommy uni stuff\\Project\\Data\\logs\\{atom1}{atom2}\\{n1_0}{orbs[l1_0]}{j1_0} {n2_0}{orbs[l2_0]}{j2_0}\\run logs\\Basis Analysis\\'
if not os.path.exists(runlog_datapath):
    os.makedirs(runlog_datapath)



    n1_0  l1_0  j1_0  n2_0  l2_0  j2_0  n1_1  l1_1  j1_1  n2_1  ...  \
0     45     0   0.5    47     0   0.5    45     1   1.5    46  ...   
1     46     0   0.5    48     0   0.5    46     1   1.5    47  ...   
2     48     0   0.5    51     0   0.5    48     1   1.5    50  ...   
3     59     0   0.5    57     0   0.5    58     1   0.5    57  ...   
4     61     0   0.5    65     0   0.5    61     1   0.5    64  ...   
5     68     0   0.5    67     0   0.5    67     1   0.5    67  ...   
6     69     0   0.5    68     0   0.5    68     1   0.5    68  ...   
7     71     0   0.5    69     0   0.5    70     1   1.5    69  ...   
8     72     0   0.5    70     0   0.5    71     1   1.5    70  ...   
9     72     0   0.5    75     0   0.5    72     1   0.5    74  ...   
10    73     0   0.5    76     0   0.5    73     1   0.5    75  ...   
11    77     0   0.5    81     0   0.5    77     1   1.5    80  ...   
12    81     0   0.5    78     0   0.5    80     1   0.5    78  ...   
13    

In [11]:
# interaction energy calculation
import time

datapath_basis = fr'E:\\Tommy uni stuff\\Project\\Data\\interesting channels\\{atom1}{atom2}\\{n1_0}{orbs[l1_0]}{j1_0} {n2_0}{orbs[l2_0]}{j2_0}\\Basis Analysis\\inter'
if not os.path.exists(datapath_basis):
    os.makedirs(datapath_basis)

# runlog open/create
if os.path.exists(runlog_datapath + f'Inter energy calculation.csv'):
    
    run_log = pd.read_csv(runlog_datapath + f'Inter energy calculation.csv')
else:
    run_log = pd.DataFrame({'timestamp' : [],'total time' : [], 'n_dif' : [], 'l_dif' : [], 'j_dif' : [], 'erange' : [], 'r' : [], 'B-field' : [], 'E-field' : [], 'theta' : []})

t0 = time.time()


# define kets
ket1 = pi.KetAtom(atom1, n=n1_0, l=l1_0, m=j1_0, j=j1_0)
ket2 = pi.KetAtom(atom2, n=n2_0, l=l2_0, m=j2_0, j=j2_0)


# run pairstate calculation over different energy ranges
pairs = get_system_pair_list_erange(ket1, ket2, theta, abs(defect) * eranges, r, b, e, ndif, ldif, jdif)
pairs_big = get_system_pair_list_erange(ket1, ket2, theta, abs(defect) * eranges, 5000, b, e, ndif, ldif, jdif)

system_pairs = pairs[0]
state_nums = pairs[1]
energies = pairs[2]
energies_big = pairs_big[2]
times = pairs[3]

# Calculate overlaps for each energy range to initial state
overlaps_inter = [system.get_eigenbasis().get_overlaps([ket1, ket2]) for system in system_pairs]
overlaps_inter_big = [system.get_eigenbasis().get_overlaps([ket1, ket2]) for system in pairs_big[0]]

print(overlaps_inter)
# Calculate interaction energy for each energy range - from the eigenstate with the largest overlap with the initial state
filtered_energies = [ energies[i][list(overlaps_inter[i]).index(max(overlaps_inter[i], key=abs))] - energies_big[i][list(overlaps_inter_big[i]).index(max(overlaps_inter_big[i], key=abs))] for i in range(len(energies)) ]

# logs - save
timestamp=time.strftime("%Y-%m-%d %H:%M:%S")

run_log_single = {'timestamp' : timestamp,'total time' : pairs[3], 'n_dif' : ndif, 'l_dif' : ldif, 'j_dif' : jdif, 'erange' : str(eranges), 'r' : r, 'B-field' : b, 'E-field' : e, 'theta' : theta}

run_log = pd.concat([run_log, pd.DataFrame([run_log_single])], ignore_index=True)

pd.DataFrame(run_log).to_csv(runlog_datapath + f'Inter energy calculation.csv', index=False)

# data - save
data = {'eranges' : eranges, 'energies' : filtered_energies, 'times' : times, 'state nums' : state_nums}
pd.DataFrame(data).to_csv(datapath_basis + run_tag + '.csv', index=False)

# Clearing memory
# del run_log, run_log_single, pairs, pairs_big, system_pairs, energies, overlaps_inter, overlaps_inter_big, energy
# collected = gc.collect()


[array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.]), array([0.00000000e+00, 0.00000000e+00, 1.63280896e-07, ...,
       0.00000000e+00, 1.05075930e-08, 0.00000000e+00], shape=(4732,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(9172,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(13240,)), array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
       0.00000000e+00, 1.57962258e-15, 0.00000000e+00], shape=(16852,)), array([0.00000000e+00, 0.00000000e+00, 4.66512765e-16, ...,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00], shape=(19848,)), array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
       0.00000000e+00, 7.28343728e-16, 0.00000000e+00], shape=(22384,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(24056,)), array([0.00000000e+00, 0.00000000e+00, 3.51986118e-14, ...,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00], shape=(25180,)), arr

PermissionError: [Errno 13] Permission denied: 'E:\\\\Tommy uni stuff\\\\Project\\\\Data\\\\interesting channels\\\\RbCs\\\\68s0.5 67s0.5\\\\Basis Analysis\\\\inter\\r 5um, theta 0deg, b 0G, nljdiff422, erange 90000.0MHz.csv'

In [29]:
pd.DataFrame(run_log).to_csv(runlog_datapath + f'Inter energy calculation.csv', index=False)

# data - save
data = {'eranges' : eranges, 'energies' : filtered_energies, 'times' : times, 'state nums' : state_nums}
pd.DataFrame(data).to_csv(datapath_basis + run_tag + '.csv', index=False)

In [13]:
print(max(overlaps_inter[1], key=abs))
print(overlaps_inter[1])
pd.DataFrame(data).to_csv(datapath_basis + run_tag + '.csv', index=False)


0.4515002760895798
[0.00000000e+00 0.00000000e+00 1.63280896e-07 ... 0.00000000e+00
 1.05075930e-08 0.00000000e+00]
